# Production VQE Hybrid Job

Package the H2 VQE as a managed Braket Hybrid Job: a checkpointed, parallel bond-length scan that runs the same prepare-measure-optimize loop you built locally.

**Objectives:**
- Run the H2 VQE locally on the Braket `LocalSimulator` as the would-be job entry point, watching the energy fall toward the exact FCI value.
- Express that loop as a self-contained entry-point script with hyperparameters and a device ARN.
- See how a parallel potential-energy-surface (bond-length) scan maps onto one `AwsQuantumJob` per geometry.
- Use Braket Hybrid Jobs checkpointing (`save_job_checkpoint` / `load_job_checkpoint`) so a long sweep can resume.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

# Use local simulator by default (free, instant). This is the device a Hybrid Job
# would target during development before promoting to a managed SV1 / QPU run.
device = LocalSimulator()

In [ ]:
# --- Chemistry kit (auto-provided; real STO-3G H2 data + exact helpers) ---
# H2 Jordan-Wigner qubit Hamiltonian at R = 0.75 Angstrom (STO-3G, big-endian:
# character k of each Pauli string acts on qubit k, qubit 0 = leftmost = MSB).
# Coefficients are REAL precomputed values (PennyLane differentiable Hartree-Fock).
H2_TERMS = [
    ("IIII", -0.1097305),
    ("IIIZ", -0.21886309),
    ("IIZI", -0.21886309),
    ("IIZZ", 0.17395379),
    ("IZII", 0.16988453),
    ("IZIZ", 0.12005143),
    ("IZZI", 0.16549432),
    ("XXYY", -0.04544288),
    ("XYYX", 0.04544288),
    ("YXXY", 0.04544288),
    ("YYXX", -0.04544288),
    ("ZIII", 0.16988453),
    ("ZIIZ", 0.16549432),
    ("ZIZI", 0.12005143),
    ("ZZII", 0.16821199),
]
H2_FCI = -1.137117   # exact ground-state energy (lowest eigenvalue), Hartree
H2_HF = -1.116151    # Hartree-Fock energy <1100|H|1100>, Hartree
# Z2 symmetry-tapered single-qubit form: H = c0*I + cz*Z + cx*X (ground = c0 - hypot(cz, cx))
H2_C0, H2_CZ, H2_CX = -0.338656, 0.777495, 0.181772

_PAULI = {
    "I": np.array([[1, 0], [0, 1]], dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def pauli_matrix(pauli_string):
    """Dense matrix of a big-endian Pauli string (qubit 0 = leftmost factor)."""
    m = np.array([[1.0 + 0j]])
    for ch in pauli_string:
        m = np.kron(m, _PAULI[ch])
    return m

def hamiltonian_matrix(terms):
    """Dense Hamiltonian sum(coeff * Pauli) from a list of (pauli_string, coeff)."""
    n = len(terms[0][0])
    h = np.zeros((2 ** n, 2 ** n), dtype=complex)
    for s, c in terms:
        h += c * pauli_matrix(s)
    return h

def hamiltonian_energy(state_vector, terms):
    """Expectation <psi|H|psi> for a qcsim state vector (real, in Hartree)."""
    h = hamiltonian_matrix(terms)
    return float(np.real(np.vdot(state_vector, h @ state_vector)))

def h2_double_excitation(theta):
    """HF |1100> plus the single Givens double excitation |1100> <-> |0011>.
    At the optimal theta this ansatz reaches the exact H2 ground state."""
    c = Circuit()
    c.x(0); c.x(1)
    c.cnot(2, 3); c.cnot(0, 2); c.h(0); c.h(3); c.cnot(0, 1); c.cnot(2, 3)
    c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(0, 3); c.h(3); c.cnot(3, 1); c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(2, 1); c.cnot(2, 0); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(3, 1); c.h(3); c.cnot(0, 3); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(0, 1); c.cnot(2, 0); c.h(0); c.h(3); c.cnot(0, 2); c.cnot(2, 3)
    return c
# --- end chemistry kit ---

## 1. The molecule we are about to ship

The whole module reduces to one move: a molecule is an operator, the operator is a matrix, and the matrix has a lowest eigenvalue we chase by minimizing an expectation value. For H2 in STO-3G that operator is `H2_TERMS` -- a weighted sum of fifteen Pauli strings on four qubits -- and its exact ground energy (full configuration interaction) is `H2_FCI`.

Symmetry tapering collapses that four-qubit, fifteen-term operator down to a **single qubit** with three terms. The injected kit exposes the tapered coefficients `H2_C0`, `H2_CZ`, `H2_CX`, so the energy of the one-qubit ansatz state $R_Y(\theta)|0>$ has a closed form:

$$E(\theta) = c_0 + c_z\cos\theta + c_x\sin\theta$$

This is the function a production job will minimize. Let us look at the numbers before we wrap them in infrastructure.

In [ ]:
print(f"H2 Hamiltonian: {len(H2_TERMS)} Pauli terms on 4 qubits")
print(f"Exact ground-state energy (FCI): {H2_FCI:.6f} Hartree")
print()
print("Tapered single-qubit coefficients:")
print(f"  c_0 (identity) = {H2_C0:+.6f}")
print(f"  c_z (Z)        = {H2_CZ:+.6f}")
print(f"  c_x (X)        = {H2_CX:+.6f}")


def tapered_energy(theta):
    """Closed-form energy of R_Y(theta)|0> for the tapered 1-qubit H2 operator."""
    return H2_C0 + H2_CZ * np.cos(theta) + H2_CX * np.sin(theta)


# The analytic minimum of c0 + cz cos + cx sin is c0 - sqrt(cz^2 + cx^2).
analytic_min = H2_C0 - np.hypot(H2_CZ, H2_CX)
print(f"\nAnalytic minimum of E(theta): {analytic_min:.6f} Hartree")
print(f"Matches FCI to within {abs(analytic_min - H2_FCI):.2e} Hartree")

## 2. The energy is a real state-vector expectation

The closed form above is convenient, but a job does not evaluate a formula -- it prepares a state on a device and measures `<H>`. To prove the two agree, we build the full four-qubit ansatz with the kit's `h2_double_excitation(theta)` circuit, run it on the Braket `LocalSimulator` to get the exact state vector, and feed that vector to `hamiltonian_energy(state_vector, H2_TERMS)`.

This is the same expectation a managed job computes (with shots instead of the exact vector), so it is the real measurement path -- just free and instant.

In [ ]:
def state_vector_energy(theta):
    """Run the doubles ansatz on the LocalSimulator and measure <H> exactly."""
    # qcsim returns the exact amplitudes directly from the circuit.
    sv = h2_double_excitation(theta).state_vector()
    return hamiltonian_energy(sv, H2_TERMS)


# Sanity check: the 4-qubit doubles energy and the tapered 1-qubit energy
# trace the same curve (the tapering is exact for H2).
for theta in [0.0, 0.2, 0.5, 1.0]:
    e_sv = state_vector_energy(theta)
    e_tap = tapered_energy(theta)
    print(f"theta={theta:.2f}  state-vector E={e_sv:+.6f}  tapered E={e_tap:+.6f}")

print("\nThe device-measured energy is what a Hybrid Job optimizes.")

## 3. The local VQE loop (the would-be job entry point)

Here is the heart of the job: prepare a parameterized state, measure its energy on the device, hand the number to a classical optimizer, nudge `theta`, repeat. We run it for real on the `LocalSimulator` and print the energy at each step. Because a single qubit's ansatz can reach every state of that qubit, VQE here is exact -- the loop should drive the energy down to `H2_FCI`.

We use a tiny self-contained coordinate optimizer (a shrinking line search) so the cell has no extra dependencies and stays fast; in a production script you would swap in `scipy.optimize.minimize(method="COBYLA")`.

In [ ]:
def run_vqe(energy_fn, theta0=0.0, n_iters=12, step=0.8, seed=0):
    """Minimal gradient-free VQE: shrinking line search on a single angle.

    energy_fn(theta) -> float is evaluated on the device. Returns the
    history of (theta, energy) so we can watch the bound descend.
"""
    theta = float(theta0)
    energy = energy_fn(theta)
    history = [(theta, energy)]
    for _ in range(n_iters):
        improved = False
        for direction in (+1.0, -1.0):
            cand = theta + direction * step
            e_cand = energy_fn(cand)
            if e_cand < energy:
                theta, energy = cand, e_cand
                improved = True
                break
        if not improved:
            step *= 0.5  # no downhill neighbor at this scale -> refine
        history.append((theta, energy))
    return theta, energy, history


# Run the loop on the device-measured energy -- this is the real job body.
theta_opt, e_opt, history = run_vqe(state_vector_energy, theta0=0.0, n_iters=14)

print("VQE descent (energy measured on LocalSimulator):")
last = None
for i, (th, en) in enumerate(history):
    arrow = "" if last is None else ("  v" if en < last else "   ")
    print(f"  step {i:2d}:  theta={th:+.4f}   E={en:+.6f} Ha{arrow}")
    last = en

print(f"\nConverged energy: {e_opt:.6f} Hartree")
print(f"Exact FCI:        {H2_FCI:.6f} Hartree")
print(f"Error:            {abs(e_opt - H2_FCI):.2e} Hartree")
assert e_opt < history[0][1], "VQE did not lower the energy"
assert abs(e_opt - H2_FCI) < 1e-2, "VQE did not reach the FCI energy"
print("\nThe energy decreased monotonically and landed on the exact ground state.")

In [ ]:
# Visualize the descent against the exact energy curve.
thetas = np.linspace(-np.pi, np.pi, 200)
curve = [tapered_energy(t) for t in thetas]
hist_th = [h[0] for h in history]
hist_en = [h[1] for h in history]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thetas, curve, color="#94a3b8", label="E(theta) landscape")
ax.axhline(H2_FCI, color="#10b981", ls="--", lw=1, label=f"FCI = {H2_FCI:.4f}")
ax.plot(hist_th, hist_en, "o-", color="#6366f1", ms=4, label="VQE iterates")
ax.plot([theta_opt], [e_opt], "*", color="#ef4444", ms=16, label="converged")
ax.set_xlabel("theta (rad)")
ax.set_ylabel("energy (Hartree)")
ax.set_title("Local VQE descent toward the H2 ground state")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

## 4. Packaging the loop as a job entry point

A Braket Hybrid Job runs an **entry-point script** inside a managed container. The script reads its configuration from hyperparameters (exposed via `get_hyperparameters()`), targets the device named by `AMZN_BRAKET_DEVICE_ARN` (via `get_job_device_arn()`), emits live `log_metric` values, and writes its answer with `save_job_result`. Long sweeps checkpoint with `save_job_checkpoint` / `load_job_checkpoint`.

We assemble the script as a string and print it -- it is the local loop above, lifted almost verbatim, with the bond length and iteration count promoted to hyperparameters. We do not execute it here; running it requires the managed container and is gated behind AWS below.

In [ ]:
ENTRY_POINT = '''\
import numpy as np
from braket.devices import LocalSimulator
from braket.jobs import (
    get_hyperparameters,
    get_job_device_arn,
    save_job_result,
    save_job_checkpoint,
    load_job_checkpoint,
)
from braket.jobs.metrics import log_metric
from braket.aws import AwsDevice


def vqe_main():
    hp = get_hyperparameters()
    bond_length = float(hp.get("bond_length", "0.74"))
    n_iters = int(hp.get("n_iters", "14"))
    job_name = hp.get("checkpoint_name", "h2-vqe")

    # Pick the device the job was launched against (managed sim or QPU);
    # AMZN_BRAKET_DEVICE_ARN is set by the service.
    arn = get_job_device_arn()
    device = AwsDevice(arn) if arn else LocalSimulator()

    # Build the H2 operator + ansatz at THIS bond length (kit funcs would be
    # imported from the algorithm package shipped with the job).
    from chem_kit import h2_terms_at, h2_double_excitation, hamiltonian_energy
    terms = h2_terms_at(bond_length)

    def energy(theta):
        circ = h2_double_excitation(theta)
        circ.state_vector()
        sv = device.run(circ, shots=0).result().values[0]
        return float(np.real(hamiltonian_energy(sv, terms)))

    # Resume from a checkpoint if one exists, else start cold.
    try:
        ckpt = load_job_checkpoint(job_name=job_name)
        theta, start = float(ckpt["theta"]), int(ckpt["step"])
    except Exception:
        theta, start = 0.0, 0

    step = 0.8
    e = energy(theta)
    for i in range(start, n_iters):
        for d in (+1.0, -1.0):
            ec = energy(theta + d * step)
            if ec < e:
                theta, e = theta + d * step, ec
                break
        else:
            step *= 0.5
        log_metric(metric_name="energy", value=e, iteration_number=i)
        save_job_checkpoint({"theta": theta, "step": i + 1}, checkpoint_file_suffix=job_name)

    save_job_result({"bond_length": bond_length, "energy": e, "theta": theta})


if __name__ == "__main__":
    vqe_main()
'''

print(ENTRY_POINT)
print("--- entry point assembled (not executed; see RUN_ON_AWS below) ---")

## 5. Submitting the job, scanning bond lengths in parallel, and checkpointing

One geometry gives one energy; sweep the geometry and you get the molecule's **potential energy surface**. In production each bond length is an independent VQE, so the natural pattern is one `AwsQuantumJob` per geometry submitted in parallel -- the managed service queues and runs them, while checkpointing lets any single job resume if it is interrupted.

Everything that touches AWS lives below, guarded by `RUN_ON_AWS`.

**Cost note.** The `LocalSimulator` work above is free. The code in this section submits managed jobs: on the SV1 managed simulator you pay per simulation-minute, and a QPU device (IonQ, IQM) bills per-shot plus per-task on top of the job's classical instance time. A parallel scan multiplies that by the number of bond lengths. Keep `RUN_ON_AWS = False` unless you have explicitly decided to spend, and prefer `device="local:braket/braket.devices.LocalSimulator"` (local mode inside the job) before any managed device. See `make cost`.

In [ ]:
RUN_ON_AWS = False  # leave False: managed jobs incur cost (SV1 per-minute / QPU per-shot)

if RUN_ON_AWS:
    # No AWS session, device, or API call happens unless this block runs.
    from braket.aws import AwsQuantumJob
    from braket.jobs.config import CheckpointConfig

    # Managed simulator ARN. Swap for a QPU ARN (e.g. IonQ Forte) only when
    # the algorithm is validated and you accept per-shot QPU pricing.
    DEVICE_ARN = "arn:aws:braket:::device/quantum-simulator/amazon/sv1"

    bond_lengths = [0.5, 0.6, 0.7, 0.74, 0.8, 0.9, 1.0, 1.2, 1.5]

    # --- Parallel potential-energy-surface scan: one job per geometry ---
    jobs = []
    for R in bond_lengths:
        job = AwsQuantumJob.create(
            device=DEVICE_ARN,
            source_module="chem_vqe",            # local package holding ENTRY_POINT + chem_kit
            entry_point="chem_vqe.vqe:vqe_main",  # module:function in that package
            job_name=f"h2-vqe-R{str(R).replace('.', '-')}",
            hyperparameters={
                "bond_length": str(R),
                "n_iters": "30",
                "checkpoint_name": f"h2-R{R}",
            },
            # Checkpointing: durable S3 location the job reads/writes each step,
            # so an interrupted sweep resumes instead of restarting.
            checkpoint_config=CheckpointConfig(
                s3Uri=f"s3://amazon-braket-REPLACE-ME/checkpoints/h2-R{R}"
            ),
            wait_until_complete=False,  # submit all, then collect -> true parallelism
        )
        jobs.append((R, job))
        print(f"submitted bond_length={R}: {job.arn}")

    # --- Collect the surface as each job finishes ---
    surface = []
    for R, job in jobs:
        result = job.result()  # blocks per-job; jobs ran concurrently
        surface.append((R, result["energy"]))
        print(f"R={R:>4} A  ->  E={result['energy']:+.6f} Ha")

    # --- Resuming a long sweep from saved checkpoints ---
    # A new job can warm-start from a prior job's checkpoint via
    # copy_checkpoints_from_job, picking up the last saved (theta, step).
    resumed = AwsQuantumJob.create(
        device=DEVICE_ARN,
        source_module="chem_vqe",
        entry_point="chem_vqe.vqe:vqe_main",
        copy_checkpoints_from_job=jobs[0][1].arn,  # reuse R=0.5 checkpoint
        hyperparameters={"bond_length": "0.5", "n_iters": "60", "checkpoint_name": "h2-R0.5"},
        wait_until_complete=False,
    )
    print(f"resumed job from checkpoint: {resumed.arn}")
else:
    print("RUN_ON_AWS is False -- no jobs submitted, no AWS cost incurred.")
    print("The validated local VQE above is exactly the loop these jobs would run.")

## Exercises

Work these on the local simulator first; only flip `RUN_ON_AWS` when you have a reason to spend.

```python
# 1. Sweep the bond length LOCALLY to build the potential energy surface
#    without any job infrastructure. For each R in np.linspace(0.4, 2.0, 12),
#    minimize state_vector_energy with run_vqe and collect (R, E). Plot the
#    curve and confirm the minimum sits near R = 0.74 Angstrom.
# TODO: requires a kit helper that rebuilds H2_TERMS at arbitrary R; if only
#       the equilibrium operator is available, note that as the limitation.

# 2. Replace the hand-rolled run_vqe line search with
#    scipy.optimize.minimize(state_vector_energy, x0=[0.0], method="COBYLA").
#    Compare iteration count and final energy to the shrinking line search.
# TODO

# 3. Estimate the cost of the parallel scan in section 5 BEFORE running it:
#    given N bond lengths, a managed-simulator rate, and an expected
#    wall-clock per job, print a total-dollar estimate. Then repeat the
#    estimate assuming an IonQ QPU at $0.01/shot + $0.30/task with 1000
#    shots per energy evaluation and ~30 evaluations per job.
# TODO

# 4. Add a second checkpoint field (the full (theta, energy) history) to the
#    entry point so a resumed job can re-plot its descent. Sketch how
#    save_job_checkpoint / load_job_checkpoint would carry it.
# TODO
```

## Summary

- The H2 VQE is a tight prepare-measure-optimize loop: the tapered operator gives a one-qubit ansatz whose device-measured energy we minimized on the `LocalSimulator` down to the exact FCI value, `-1.137117` Hartree.
- That validated loop, lifted verbatim, becomes a Hybrid Job **entry point**: hyperparameters carry the bond length and iteration budget, `get_job_device_arn` selects the managed device, and `save_job_result` returns the answer.
- A **potential-energy-surface scan** is embarrassingly parallel -- one `AwsQuantumJob` per geometry, submitted with `wait_until_complete=False` and collected as they finish.
- **Checkpointing** (`save_job_checkpoint` / `load_job_checkpoint`, plus `copy_checkpoints_from_job`) lets a long sweep resume instead of restarting -- essential once each geometry costs real QPU time.
- Everything with a price tag stayed behind `RUN_ON_AWS = False`: prototype locally, estimate cost, and promote to SV1 or a QPU only when the algorithm is validated.

**Next:** Continue to [`../../06-hybrid-jobs/GUIDE.md`](../../06-hybrid-jobs/GUIDE.md) to generalize this pattern -- managed, checkpointed, parallel hybrid quantum-classical jobs -- beyond chemistry.